# 08 Claude red-team: root replay, custody attacks, context ablation
Replays only PUBLIC artifacts. Model inference ran on magicSTUDIObox and MacBookPro via `redteam/ablation_runner.py`; this notebook recomputes every statistic from the public per-case bits. Hashes prove identity, not causality.

In [1]:
import json, subprocess, sys
from pathlib import Path
R = Path.cwd()
sys.path.insert(0, str(R/'redteam'))
r = subprocess.run([sys.executable, 'redteam/verify_roots.py', '--repo', '.'], capture_output=True, text=True, check=True)
rep = json.loads(r.stdout)
print('pins reproduced from repo-only data:', rep['pins_reproduced'])
print('728 root without source bytes:', rep['ln_mini_728'])

pins reproduced from repo-only data: {'LN_MINI_728': False, 'PREDECESSOR_15': True, 'REVIEW_66': True, 'REVIEW_79': True}
728 root without source bytes: {'status': 'NOT_TESTED_SOURCE_UNAVAILABLE'}


In [2]:
dev = json.loads((R/'redteam/receipts/ROOT_REPLAY_magicSTUDIObox.json').read_text())
print('magicSTUDIObox source-byte replay:', dev['pins_reproduced'], dev['predecessor_15']['source_bytes'])
print({k: v for k, v in dev['ln_mini_728'].items() if k != 'pinned_root'})

magicSTUDIObox source-byte replay: {'LN_MINI_728': True, 'PREDECESSOR_15': True, 'REVIEW_66': True, 'REVIEW_79': True} PASS
{'assistant_turns_in_public_inputs': 0, 'inclusion_proofs_verified_against_pin': '728/728', 'input_records_identical_to_disk': '728/728', 'leaves': 729, 'odd_leaf_count_duplicate_ambiguity': 'PRESENT', 'oracle_marker_strings_in_public_files': 0, 'recipe_sha_vs_manifest': 'PASS', 'recomputed_root': '5f9a7e51f1ee03d70d34e99358fe14230efa396f7fd851df21f424e6843f61b1', 'root_vs_manifest': 'PASS', 'root_vs_pin': 'PASS', 'rows': 728, 'snapshot_id_rederived': 'PASS', 'source_fco_id_vs_manifest': 'PASS', 'source_sha256': '544e0bf2975b04cf1d2be6654b4baccbe04db3cf4f167d80c81e25a9fc3fe72c', 'source_sha_vs_manifest': 'PASS'}


In [3]:
r = subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_redteam_attacks', '-v'], capture_output=True, text=True)
print(r.stderr[-2500:]); assert r.returncode == 0

tests.test_redteam_attacks.CrossImplementation.test_defence_domain_separation) ... ok
test_defence_independent_impl_matches_on_random_trees (tests.test_redteam_attacks.CrossImplementation.test_defence_independent_impl_matches_on_random_trees) ... ok
test_defence_independent_impl_matches_predecessor_on_real_bundle (tests.test_redteam_attacks.CrossImplementation.test_defence_independent_impl_matches_predecessor_on_real_bundle) ... ok
test_defence_parent_inside_context_leaf_is_committed (tests.test_redteam_attacks.CustodyAttacks.test_defence_parent_inside_context_leaf_is_committed) ... ok
test_weakness_dangling_fcg_edge_accepted (tests.test_redteam_attacks.CustodyAttacks.test_weakness_dangling_fcg_edge_accepted) ... ok
test_weakness_duplicate_json_key_parser_differential (tests.test_redteam_attacks.CustodyAttacks.test_weakness_duplicate_json_key_parser_differential)
A leaf line with two sha256 keys: Python keeps the last; a first-wins parser sees the forged one. ... ok
test_weakness_dupli

In [4]:
subprocess.run([sys.executable, 'redteam/analyze.py', 'report', '--results', 'redteam/results'], check=True)
print((R/'redteam/results/EVIDENCE_MATRIX.md').read_text())

# Evidence matrix (generated by redteam/analyze.py report)

Strict parser (identical to predecessor). Abstentions count as wrong. AUROC NOT_COMPUTED (letter-only outputs).

## Accuracy per arm

| lane | partition | arm | n | correct | abstain | lenient correct | median s | prompt tok |
|---|---|---|---|---|---|---|---|---|
| P-L1 | discovery | B0_baseline | 38 | 2 | 35 | 6 | 0.80 | 21732 |
| P-L1 | discovery | C1_fco_projection | 38 | 6 | 28 | 6 | 1.02 | 29023 |
| P-L1 | discovery | C2_full_fcg | 38 | 0 | 37 | 3 | 2.33 | 73819 |
| P-L1 | discovery | C3_minus_source_leaf | 38 | 3 | 34 | 4 | 2.92 | 66518 |
| P-L1 | discovery | C4_corrupt_leaf | 38 | 0 | 37 | 3 | 2.74 | 73828 |
| P-L1 | discovery | C5_edge_changed | 38 | 0 | 37 | 3 | 3.30 | 73857 |
| P-L1 | discovery | C6_prompt_injection | 38 | 0 | 38 | 4 | 2.83 | 75187 |
| P-L1 | discovery | C7_gold_leak_positive_control | 38 | 3 | 32 | 4 | 3.28 | 74009 |
| P-L1 | discovery | R0_cold_replay_baseline | 38 | 2 | 35 | 6 | 0.79 | 21732 |
| 

In [5]:
m = json.loads((R/'redteam/results/EVIDENCE_MATRIX.json').read_text())
for a in m['admission']: print(a)
assert not any(a['admitted'].startswith('ADMIT') for a in m['admission'] if a['comparison'].startswith('K'))

{'admitted': 'NOT_ADMITTED', 'comparison': 'H1', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE', 'lane': 'P-L1'}
{'admitted': 'NOT_ADMITTED', 'comparison': 'H1', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE', 'lane': 'P-L3'}
{'admitted': 'NOT_ADMITTED', 'comparison': 'H1', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE', 'lane': 'S-L0'}
{'admitted': 'NOT_ADMITTED', 'comparison': 'H1', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE', 'lane': 'S-L1'}
{'admitted': 'NOT_ADMITTED', 'comparison': 'H2', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE', 'lane': 'P-L1'}
{'admitted': 'NOT_ADMITTED', 'comparison': 'H2', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE', 'lane': 'P-L3'}
{'admitted': 'NOT_ADMITTED', 'comparison': 'H2', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE', 'lane': 'S-L0'}
{'admitted': 'NOT_ADMITTED', 'comparison': 'H2', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE', 'lane': 'S-L1'}
{'admitted': 'NOT_ADMITTED', 'comparison': 'H3', 'discovery': 'NEGATIVE', 'heldout': 'NEGATIVE',

In [6]:
iv = json.loads(subprocess.run([sys.executable, 'redteam/impact_view.py', '.'], capture_output=True, text=True, check=True).stdout)
print(json.dumps(iv['arm10_changed_source_version'], indent=1))

{
 "declared_downstream_via_fcg_edges": [
  {
   "from": "datasets/longebench/mini/eval-00000-of-00001.parquet",
   "relation": "DERIVED_TO",
   "to": "custody/snapshots/LN-MINI-ec719d04c3707ca1cb/manifest.json"
  }
 ],
 "earliest_changed_leaf": {
  "index": 1,
  "path": "datasets/longebench/mini/eval-00000-of-00001.parquet"
 },
 "finding": "impact view under-reports: discovery outputs, split manifest and statistics consume the parquet but FCG_EDGES declares only parquet->snapshot manifest; a source revision would not be propagated to them by graph traversal",
 "outputs_that_depend_on_source_but_have_no_declared_edge": [
  "eval/lb_mini_discovery_lfm2-1.2b-longevity-q4km_latest.jsonl",
  "eval/insilico_qwen3_06b_discovery_attempt3.jsonl",
  "eval/paired_discovery_mcnemar_bootstrap.json",
  "eval/longebench_split_manifest.json"
 ],
 "predecessor_root_retained": "9e4d4df8aa7a887ecc8b792a1d099832b584c16f81ea821897fcd437f6fd3a9a",
 "simulated_successor_root": "83af1baa845005a939a7b03e24f6a